# 14장 파일 입출력
본 노트북은 C 언어의 **파일 입출력 기법**의 개념을 정리하고, 실습 예제 코드를 분석합니다.  
본 학습을 통해 파일과 스트림의 개념을 이해하고, 텍스트 및 이진 파일 입출력, 오류 처리, 임의 접근(Random Access) 방식을 활용할 수 있게 됩니다.

---

## 14.1 파일 열기와 닫기

### 1. 파일과 스트림(Stream)
- **파일(File)**: 자료를 디스크(보조기억장치)에 저장하기 위한 논리적인 단위입니다.
- **스트림(Stream)**: 프로그램과 입출력 장치 사이에 데이터가 이동하는 논리적인 연결 채널입니다.
- C 표준 라이브러리에서는 장치 종류에 상관없이 일관된 입출력 함수를 사용할 수 있도록 스트림을 사용합니다.

### 2. 표준 스트림
C 프로그램 실행 시 운영체제에 의해 자동으로 생성되고 관리되는 3가지 표준 스트림이 있습니다.
1. **`stdin`** (표준 입력 스트림): 기본적으로 키보드를 가리킵니다.
2. **`stdout`** (표준 출력 스트림): 기본적으로 모니터 화면을 가리킵니다.
3. **`stderr`** (표준 오류 스트림): 오류 메시지를 출력하며 기본적으로 모니터를 가리킵니다.

### 3. 파일 포인터와 `FILE` 구조체
- 파일 입출력을 처리할 때는 `<stdio.h>`에 정의된 `FILE` 구조체를 가리키는 **파일 포인터(`FILE *`)**를 사용합니다.
- `FILE` 구조체에는 파일 크기, 버퍼링 상태, 현재 파일 읽기/쓰기 위치(파일 위치 지시자) 등의 다양한 제어 정보가 들어있습니다.

### 4. 파일 열기 함수 비교

#### `fopen`
- **원형**: `FILE *fopen(const char *filename, const char *mode);`
- **설명**: `filename`으로 지정한 파일을 `mode` 접근 모드로 열어 유효한 `FILE` 포인터를 반환합니다. 실패 시 `NULL`을 반환합니다.

#### `fopen_s` (C11 Annex K 안전 기능)
- **원형**: `errno_t fopen_s(FILE **pFile, const char *filename, const char *mode);`
- **설명**: 파일 포인터를 매개변수로 직접 받아 넘기며, 성공 시 `0`을, 실패 시 에러 코드를 반환합니다.
- > **주의**: `fopen_s`는 macOS의 Clang 컴파일러 등 일부 개발 환경에서 지원하지 않을 수 있어, 맥 환경에서는 주로 표준 `fopen` 함수를 사용합니다.

### 5. 파일 접근 모드 종류

| 모드 | 동작 설명 | 파일이 없을 때 | 파일이 있을 때 |
| :--- | :--- | :--- | :--- |
| **`"r"`** / **`"rb"`** | 읽기 전용 (텍스트 / 이진) | `NULL` 반환 (실패) | 정상 처리 |
| **`"w"`** / **`"wb"`** | 쓰기 전용 (텍스트 / 이진) | 새로운 파일 생성 | 이전 내용 삭제 후 새로 작성 |
| **`"a"`** / **`"ab"`** | 추가 쓰기 (텍스트 / 이진) | 새로운 파일 생성 | 기존 내용 끝에 추가 기록 |
| **`"r+"`** / **`"r+b"`**| 읽기/쓰기 혼용 | `NULL` 반환 (실패) | 기존 내용을 수정 |
| **`"w+"`** / **`"w+b"`**| 읽기/쓰기 혼용 | 새로운 파일 생성 | 이전 내용 삭제 후 새로 작성 |
| **`"a+"`** / **`"a+b"`**| 읽기/추가쓰기 혼용 | 새로운 파일 생성 | 기존 내용 끝에 추가 기록 |

### 6. 파일 닫기 함수
- **원형**: `int fclose(FILE *fp);`
- **설명**: 열려있는 파일 스트림과의 연결을 끊고, 출력 버퍼에 남아있던 데이터를 디스크에 물리적으로 기록(Flush)합니다. 성공 시 `0`을, 오류 발생 시 `EOF`(-1)를 반환합니다.

## 14.2 텍스트 파일 입출력

텍스트 파일은 줄바꿈 문자(`\n`)를 운영체제에 맞춰 자동 변환하여 읽고 씁니다. (예: Windows에서는 `\r\n`으로 쓰고 읽을 때 `\n`으로 자동 변환)

### 1. 문자 단위 입출력 (`fgetc`, `fputc`)
- `int fgetc(FILE *fp);` : 파일에서 한 문자를 읽어 정수형(`int`)으로 반환합니다. 파일 끝에 도달하면 `EOF`(-1)를 반환합니다.
- `int fputc(int c, FILE *fp);` : 한 문자 `c`를 파일에 씁니다. 성공 시 해당 문자를 반환하고 실패 시 `EOF`를 반환합니다.

---

### 예제 14.1: fopen_s 방식의 문자 단위 쓰기/읽기 (`04_file_io/ex14_1.c`)
문자열 상수를 선언한 뒤 파일 `test.dat`에 문자 단위로 쓴 후, 다시 읽어와서 화면에 출력합니다.  
> **학습 포인트**: `fgetc` 함수의 반환값을 담는 변수 `c`는 **`int`형**이어야 `EOF`(-1) 판별을 정확히 할 수 있습니다.

In [ ]:
// 04_file_io/ex14_1.c
#include <stdio.h>
#include <stdlib.h>

int main()
{
    char *s = "나비, 제비야, 깝치지 마라.";
    int c; // fgetc()는 EOF(-1)를 반환하므로 int형으로 선언해야 정확히 체크됩니다.
    FILE *fp;

    // 1. 파일 쓰기 모드로 열기
    // Visual Studio: fopen_s(&fp, "test.dat", "w");
    fp = fopen("test.dat", "w");
    if (fp != NULL)
        printf("파일 'test.dat'는 쓰기 위하여 열렸습니다.\n");
    else {
        printf("파일 'test.dat'는 쓰기 위하여 열리지 않았습니다.\n");
        exit(1);
    }

    // 문자 단위로 파일 쓰기
    while (*s) {
        if (fputc(*s++, fp) == EOF) {
            printf(" 파일쓰기 오류\n");
            exit(1);
        }
    }
    fclose(fp);

    // 2. 파일 읽기 모드로 열기
    fp = fopen("test.dat", "r");
    if (fp != NULL)
        printf("파일 'test.dat'는 읽기 위하여 열렸습니다.\n");
    else {
        printf("파일 'test.dat'는 읽기 위하여 열리지 않았습니다.\n");
        exit(1);
    }

    // 문자 단위로 읽어 콘솔 화면 출력
    while ((c = fgetc(fp)) != EOF)
        putchar(c);
    printf("\n");
    fclose(fp);

    return 0;
}

### 예제 14.1.1: 표준 fopen 함수 사용 버전 (`04_file_io/ex14_1_1.c`)
기존 `ex14_1.c`와 비교하여 `fopen`의 반환값 `NULL` 검사 패턴을 확인하세요.

In [ ]:
// 04_file_io/ex14_1_1.c
#include <stdio.h>
#include <stdlib.h>

int main()
{
    char *s = "나비, 제비야, 깝치지 마라.";
    int c;
    FILE *fp;

    fp = fopen("test.dat", "w");
    if (fp == NULL) {
        printf("파일 'test.dat'는 쓰기 위하여 열리지 않았습니다.\n");
        exit(1);
    }
    else
        printf("파일 'test.dat'는 쓰기 위하여 열렸습니다.\n");

    while (*s){
        if (fputc(*s++, fp) == EOF) {
            printf(" 파일쓰기 오류\n");
            exit(1);
        }
    }
    fclose(fp);

    fp = fopen("test.dat", "r");
    if (fp == NULL) {
        printf("파일 'test.dat'는 읽기 위하여 열리지 않았습니다.\n");
        exit(1);
    }
    else
        printf("파일 'test.dat'는 읽기 위하여 열렸습니다.\n");

    while ((c = fgetc(fp)) != EOF)
        putchar(c);
    printf("\n");
    fclose(fp);

    return 0;
}

### 예제 14.2: 문자 단위 파일 복사 (`04_file_io/ex14_2.c`)
자신(`ex14_2.c`)을 입력 스트림으로 읽어들여 다른 임시 파일(`temp.c`)로 한 바이트씩 고스란히 저장합니다.

In [ ]:
// 04_file_io/ex14_2.c
#include <stdio.h>

int main()
{
    FILE *fp1, *fp2;
    int c;

    fp1 = fopen("ex14_2.c", "r");
    fp2 = fopen("temp.c", "w");

    if (fp1 == NULL || fp2 == NULL) {
        puts("파일 열기 실패");
        return 1;
    }

    // 복사 수행
    while ((c = fgetc(fp1)) != EOF) 
        fputc(c, fp2);

    fclose(fp1);
    fclose(fp2);

    printf("ex14_2.c 파일을 temp.c 파일로 복사 하였습니다.\n");
    return 0;
}

### 2. 문자열 단위 입출력 (`fgets`, `fputs`)
- `char *fgets(char *s, int n, FILE *fp);` : 파일로부터 최대 `n-1`개의 문자열을 읽어 `s`에 저장하고 문자열의 끝에 `\0`를 덧붙입니다. 개행 문자(`\n`)를 만나거나 파일 끝에 이르면 종료됩니다. 성공 시 버퍼의 주소를, 오류/EOF 시 `NULL`을 리턴합니다.
- `int fputs(const char *s, FILE *fp);` : `\0` 전까지의 문자열을 파일에 기록합니다. (개행 문자를 자동 추가하지 않습니다.) 성공 시 음수가 아닌 값을, 실패 시 `EOF`를 리턴합니다.
- **장점**: 기존 `gets`는 버퍼 오버플로우 취약점이 존재하여 표준 사용이 제한되지만, `fgets`는 최대 길이를 강제하므로 안전한 코딩 기법입니다.

---

### 예제 14.3: 문자열 단위 파일 쓰기/읽기 (`04_file_io/ex14_3.c`)
1줄의 문자열을 `fputs`로 파일에 기록하고, `fgets`를 사용하여 다시 가져와서 화면에 출력합니다.

In [ ]:
// 04_file_io/ex14_3.c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    FILE *fp;
    char line[20] = "Hello! World.";

    // 1. 파일 쓰기
    fp = fopen("test.dat", "w");
    if (fp == NULL)
    {
        puts("File open error!");
        exit(1);
    }
    
    if (fputs(line, fp) == EOF)
        puts("fputs() error");
    else
        fputc('\n', fp);
        
    puts("파일이 정상적으로 생성되었습니다.");
    fclose(fp);

    // 2. 파일 읽기
    fp = fopen("test.dat", "r");
    if (fp == NULL)
    {
        puts("File open error!");
        exit(1);
    }
    
    if (fgets(line, 20, fp) == NULL)
        puts("fgets() error");
    else {
        puts("읽은 파일의 내용은 : ");
        printf("%s", line);
    }
    fclose(fp);

    return 0;
}

### 3. 형식화된 데이터 입출력 (`fprintf`, `fscanf`)
- `int fprintf(FILE *fp, const char *format, ...);` : 형식 문자열에 서식 코드를 대응하여 파일 스트림에 형식화된 출력을 보냅니다.
- `int fscanf(FILE *fp, const char *format, ...);` : 파일에서 데이터를 읽어와서 지정한 변수들의 주소 공간에 나누어 할당합니다.
- **보안 기능 (`_s`)**: Visual Studio 등에서는 버퍼 크기를 추가 인자로 입력받는 `fscanf_s`와 포인터 체크를 강화한 `fprintf_s` 사용을 제안합니다. 맥 환경 등 이식성이 중요한 표준 환경에서는 일반 `fscanf`, `fprintf`를 올바른 너비 지정자(예: `%19s`)와 함께 사용합니다.

---

### 예제 14.4: 형식 지정자를 이용한 입출력 (`04_file_io/ex14_4.c`)
여러 자료형(int, double, char[])을 정돈된 포맷(`"%20s %10d %10lf\n"`)으로 파일에 쓴 후, 다시 똑같은 포맷으로 끊어서 가져옵니다.
> **주의**: `%s` 지정자로 읽을 때는 크기를 제어하지 않으면 버퍼 폭발이 일어날 수 있으므로 `%19s` 처럼 최대 크기 제한을 두는 것이 안전합니다.

In [ ]:
// 04_file_io/ex14_4.c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int k1 = 20000, k2;
    double f1 = 625.78, f2;
    char st1[20] = "Welcome!", st2[20];
    FILE *fp;

    printf("원래의 데이터: %s, %d, %lf\n", st1, k1, f1);
    
    // 1. 파일 쓰기
    fp = fopen("test.dat", "w");
    if (fp == NULL)
    {
        printf("파일 열기 오류\n");
        exit(1);
    }
    fprintf(fp, "%20s %10d %10lf\n", st1, k1, f1);
    fclose(fp);

    // 2. 파일 읽기
    fp = fopen("test.dat", "r");
    if (fp == NULL)
    {
        printf("파일 열기 오류\n");
        exit(1);
    }
    
    // 안전한 읽기를 위해 버퍼크기 직전 크기 19까지 제한하여 %19s로 파싱합니다.
    if (fscanf(fp, "%19s %d %lf", st2, &k2, &f2) != 3) {
        printf("파일 읽기 오류\n");
        fclose(fp);
        exit(1);
    }
    printf("파일의 데이터: %s, %d, %lf\n", st2, k2, f2);
    fclose(fp);
    
    return 0;
}

### 실습문제 14.1: 키보드로부터 입력받은 파일 출력하기 (`04_file_io/px14_1.c`)
실행 후 읽을 대상 파일 경로를 입력하면, 문자 단위로 한 글자씩 가져와 콘솔 창에 똑같이 뿌려줍니다.

In [ ]:
// 04_file_io/px14_1.c
#include <stdio.h>

int main() {
    FILE *fp;
    char fname[20];
    int c;

    printf("읽을 파일의 이름을 입력 하시오 : ");
    if (scanf("%19s", fname) != 1) {
        puts("입력 오류");
        return 1;
    }

    fp = fopen(fname, "r");
    if (fp == NULL) {
        puts("파일 열기 실패");
        return 1;
    }
    
    while ((c = fgetc(fp)) != EOF) 
        putchar(c);
    putchar('\n');
    fclose(fp);
    
    return 0;
}

## 14.3 오류 처리 함수

파일 입출력 도중 발생하는 오류나 끝 도달 조건을 검사하는 함수들입니다.

### 1. `feof` (Find End Of File)
- **원형**: `int feof(FILE *fp);`
- **설명**: 파일 위치 지시자가 파일의 마지막(EOF)에 도달했는지 확인합니다. 끝에 도달했으면 **0이 아닌 값(참)**을, 아니면 `0`을 반환합니다.
- > **주의**: `feof`는 EOF를 넘어선 읽기 작업이 한 번 **실패한 직후**에 참을 반환합니다. 따라서 루프 시작 조건에서 `while (!feof(fp))`로 체크하고 무조건 바로 `fgetc`한 문자를 출력하면, 파일 끝부분에 도달했을 때 에러 값(-1)이 한번 콘솔에 비정상적으로 출력되는 흔한 실수를 겪을 수 있습니다. 반드시 읽어들인 값의 유효성 검사 처리를 추가해야 합니다.

### 2. `ferror` (Find File Error)
- **원형**: `int ferror(FILE *fp);`
- **설명**: 입출력 수행 중 오류가 났는지 검출합니다. 오류 발생 시 **0이 아닌 값(참)**을, 정상이면 `0`을 반환합니다.

### 3. `strerror` / `strerror_s`
- **원형**: `char *strerror(int errnum);` 또는 `errno_t strerror_s(char *buf, size_t size, int errnum);`
- **설명**: `errno`에 담긴 에러 번호에 매핑되는 시스템 상세 오류 메시지 문자열을 반환하여 개발자가 디버깅하기 쉽게 도와줍니다.

---

### 예제 14.5: 파일 끝 및 오류 점검하기 (`04_file_io/ex14_5.c`)
파일 열기 에러가 났을 때 시스템 오류 설명을 출력하며, 문자 읽기 루프를 돌면서 정상 종료(`feof`)인지 파일 오류(`ferror`)인지 파악합니다.

In [ ]:
// 04_file_io/ex14_5.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <errno.h>

int main()
{
    char *s = "맨드라미 들마꽃에도 인사를 해야지.";
    int c;
    FILE *fp;

    // 1. 파일 쓰기
    fp = fopen("test.dat", "w");
    if (fp == NULL) {
        printf("파일 열기 오류\n");
        exit(1);
    }
    while (*s){
        if (fputc(*s++, fp) == EOF) {
            printf(" 파일 쓰기 오류\n");
            exit(1);
        }
    }
    fclose(fp);

    // 2. 파일 읽기 및 상세 오류 파악
    fp = fopen("test.dat", "r");
    if (fp == NULL) {
        printf("오류 : %s \n", strerror(errno));
        exit(1);
    }

    while (1) {
        c = fgetc(fp);
        if (c == EOF) {
            if (ferror(fp)) {
                printf("\n파일 읽기 중 오류 발생\n");
            }
            if (feof(fp)) {
                // 파일 끝에 정상 도달
            }
            break;
        }
        putchar(c);
    }
    printf("\n");
    fclose(fp);
    
    return 0;
}

### 예제 14.5.1: feof 검사 오류 방지 처리 (`04_file_io/ex14_5_1.c`)
교안 원본에 가깝게 작성하되, `c != EOF` 안전 처리를 더해 마지막 문자가 중복 인쇄되지 않도록 구성한 버전입니다.

In [ ]:
// 04_file_io/ex14_5_1.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <errno.h>

int main()
{
    char *s = "맨드라미 들마꽃에도 인사를 해야지.";
    int c;
    FILE *fp;

    fp = fopen("test.dat", "w");
    if (fp == NULL) {
        printf("파일 열기 오류\n");
        exit(1);
    }
    while (*s){
        if (fputc(*s++, fp) == EOF) {
            printf(" 파일 쓰기 오류\n");
            exit(1);
        }
    }
    fclose(fp);

    fp = fopen("test.dat", "r");
    if (fp == NULL) {
        printf("파일 열기 오류: %s\n", strerror(errno));
        exit(1);
    }

    while (!feof(fp)) {
        c = fgetc(fp);
        if (c != EOF) {
            putchar(c);
        }
        if (ferror(fp)) {
            printf("파일 오류\n");
            break;
        }
    }
    printf("\n");
    fclose(fp);
    
    return 0;
}

## 14.4 이진 파일 입출력

이진 파일(Binary File)은 자료를 어떠한 가공이나 가독문자 변환 없이 메모리 그대로 0과 1의 비트열 형태로 저장하고 읽어들입니다.  
**장점**: 텍스트 인코딩 변환 오버헤드가 없으므로 속도가 매우 빠르고, 부동소수점 등 실수 연산에서 데이터 정밀도 손실이 일어나지 않습니다.

### 블록 단위 입출력 (`fread`, `fwrite`)
- `size_t fread(void *buffer, size_t size, size_t count, FILE *fp);` : 파일로부터 크기가 `size` 바이트인 데이터를 `count` 개수만큼(총 `size * count` 바이트) 읽어서 `buffer` 메모리 주소 영역에 채워넣습니다.
- `size_t fwrite(const void *buffer, size_t size, size_t count, FILE *fp);` : `buffer`에 담긴 데이터 블록을 그대로 파일 스트림에 출력합니다.
- **반환값**: 두 함수 모두 **실제 입출력에 성공한 항목의 개수(`count`)**를 반환합니다. 반환된 값이 지정한 `count` 값보다 작을 경우, 파일의 끝(EOF)에 도달했거나 에러가 났음을 의미하므로 후속 점검이 필요합니다.

---

### 예제 14.6: 정수 배열 저장 및 로드 (`04_file_io/ex14_6.c`)
5개의 정수 값을 가지는 `int a[5]` 배열을 통째로 이진 쓰기(`"wb"`)로 파일에 저장하고, 다시 `int b[5]`로 읽어와 메모리 매핑 결과를 확인합니다.

In [ ]:
// 04_file_io/ex14_6.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    FILE *fp;
    int a[5] = {1, 2, 3, 4, 5};
    int i, b[5];

    // 1. 이진 쓰기 모드 ("wb")
    fp = fopen("test.dat", "wb");
    if (fp == NULL) {
        puts("쓰기 위해 파일 열기 실패");
        exit(1);
    }
    
    // 정수 5개 블록(20바이트) 일시 출력
    if (fwrite(a, sizeof(int), 5, fp) != 5) {
        puts("파일에 기록 오류");
        exit(1);
    }
    fclose(fp);

    // 2. 이진 읽기 모드 ("rb")
    fp = fopen("test.dat", "rb");
    if (fp == NULL) {
        puts("읽기 위해 파일 열기 실패");
        exit(1);
    }
    
    // 20바이트 데이터를 그대로 메모리 버퍼 b에 로드
    if (fread(b, sizeof(int), 5, fp) != 5) {
        puts("파일로부터 읽기 오류");
        exit(1);
    }
    fclose(fp);

    for (i = 0; i < 5; i++)
        printf("%d ", b[i]);
    printf("\n");
    
    return 0;
}

### 예제 14.7: 이진 스트림 블록 단위 파일 복사 (`04_file_io/ex14_7.c`)
256바이트 버퍼를 선언하여 `fread`가 리턴하는 실제 바이트 수만큼 `fwrite`함으로써 이진 구조의 파일이나 용량이 비교적 큰 파일도 왜곡 없이 빠르고 안전하게 복사할 수 있습니다.

In [ ]:
// 04_file_io/ex14_7.c
#include <stdio.h>

int main() {
    FILE *sfp, *dfp;
    char sfile[20] = {"ex14_7.c"};
    char dfile[20] = {"temp.c"};
    char buf[256];
    size_t rcnt, wcnt;

    sfp = fopen(sfile, "rb");
    dfp = fopen(dfile, "wb");
    
    if (sfp == NULL || dfp == NULL) {
        fprintf(stderr, "파일 열기 실패...\n");
        return 1;
    }
    
    // 256바이트 블록 복사 루프
    while ((rcnt = fread(buf, 1, sizeof(buf), sfp)) > 0) {
        wcnt = fwrite(buf, 1, rcnt, dfp);
        if (wcnt < rcnt) {
            fprintf(stderr, "파일 쓰기 오류...\n");
            fclose(sfp);
            fclose(dfp);
            return 1;
        }
    }
    
    printf("%s에서 %s로 복사 되었습니다\n", sfile, dfile);
    fclose(sfp);
    fclose(dfp);
    
    return 0;
}

## 14.5 파일 임의 접근 방법 (Random Access)

파일 읽기/쓰기 작업을 스트림의 처음부터 순차적으로 진행하는 **순차 접근(Sequential Access)** 방식과 달리, 파일 스트림의 중간이나 임의의 특정 위치로 파일 지시자 위치를 바로 점프하여 제어하는 방식을 **임의 접근(Random Access)**이라고 합니다.

### 1. 주요 파일 위치 제어 함수

#### `fseek` (File Seek)
- **원형**: `int fseek(FILE *fp, long offset, int origin);`
- **설명**: 파일 위치 지시자의 위치를 `origin` 기준으로부터 `offset` 바이트만큼 앞/뒤로 이동시킵니다. 성공 시 `0`을, 오류 발생 시 `0`이 아닌 값을 반환합니다.
- **기준 매크로 (`origin`)**:
  - `SEEK_SET` (0): 파일의 시작 지점 기준
  - `SEEK_CUR` (1): 파일 포인터의 현재 작업 지점 기준
  - `SEEK_END` (2): 파일의 끝 지점 기준

#### `ftell` (File Tell)
- **원형**: `long ftell(FILE *fp);`
- **설명**: 파일의 시작 지점으로부터 현재 파일 위치 지시자가 있는 지점까지의 누적 거리(바이트 오프셋 수)를 반환합니다. 실패 시 `-1L`을 반환합니다.

#### `rewind`
- **원형**: `void rewind(FILE *fp);`
- **설명**: 파일 위치 지시자를 파일의 맨 처음으로 즉시 되돌려 보냅니다. `fseek(fp, 0L, SEEK_SET);` 호출과 완전히 동일한 효과를 가집니다.

### 2. 기타 유용한 파일 처리 제어 함수
- `int remove(const char *filename);` : 해당 물리 파일을 삭제합니다.
- `int rename(const char *oldname, const char *newname);` : 파일명을 바꿉니다.
- `int fflush(FILE *fp);` : 출력 스트림인 경우 버퍼의 임시 저장 데이터를 강제로 디스크에 내보내고 버퍼를 깨끗이 비웁니다. 입력 스트림인 경우 입력 버퍼를 비워 잔여 키 입력을 소멸시킵니다.

---

### 예제 14.8: 레코드 크기 지정을 이용한 정형 임의 입출력 (`04_file_io/ex14_8.c`)
구조체 `Nobel_Literature` 객체를 파일 크기 계산 공식(`RECLEN = 86바이트`)에 맞춰 띄어쓰기를 채워 저장한 뒤 임의 접근 점프(`fseek(fp, RECLEN * recno, SEEK_SET)`)하여 지정 영역에 덮어씁니다.

In [ ]:
// 04_file_io/ex14_8.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Nobel_Literature {
    char title[41];
    char author[41];
    unsigned int year;
} NOBEL;

// title(40) + author(40) + year(6) = 86
#define RECLEN 86
#define FORMAT "%-40s%-40s%6d"
FILE *fp;

void datawrite(int recno, char *title, char *author, int year);
void dataread(NOBEL *nbook);

int main()
{
    int i;
    NOBEL nbook;

    // 1. 샘플 데이터 임의 생성
    fp = fopen("booklist.dat", "wb");
    if (fp == NULL) {
        puts("파일 쓰기 모드 열기 에러");
        exit(1);
    }
    datawrite(0, "염소의 축제", "마리오 바르가스 요사", 2010);
    datawrite(1, "기억이 나를 본다", "토마스 트란스트뢰메르", 2011);
    datawrite(2, "사부님은 갈수록 유머러스해진다", "모옌", 2012);
    fclose(fp);

    // 2. 순차적으로 레코드 읽기
    fp = fopen("booklist.dat", "rb");
    if (fp == NULL) {
        puts("파일 읽기 모드 열기 에러");
        exit(1);
    }
    for (i = 0; i < 3; i++) {
        dataread(&nbook);
        printf("%d\t제목 : %s\n\t작가 : %s\n\t수상년도 : %d \n\n", 
            i, nbook.title, nbook.author, nbook.year);
    }
    fclose(fp);
    
    return 0;
}

void datawrite(int recno, char *title, char *author, int year)
{
    fseek(fp, RECLEN * recno, SEEK_SET);
    fprintf(fp, FORMAT, title, author, year);
}

void dataread(NOBEL *nbook)
{
    fgets(nbook->title, 41, fp);
    fgets(nbook->author, 41, fp);
    if (fscanf(fp, "%d", &nbook->year) != 1) {
        // 파싱 점검
    }
}

### 실습문제 14.2: 선택적 레코드 임의 조회하기 (`04_file_io/px14_2.c`)
사용자가 터미널 창에서 보고 싶은 행 번호(0, 1, 2)를 지정하면 해당 레코드의 바이트 오프셋 계산값으로 `fseek`하여 곧바로 조회하여 상세 정보를 출력합니다.

In [ ]:
// 04_file_io/px14_2.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Nobel_Literature {
    char title[41];
    char author[41];
    unsigned int year;
} NOBEL;

#define RECLEN 86
#define FORMAT "%-40s%-40s%6d"
FILE *fp;

void datawrite(int recno, char *title, char *author, int year);
void dataread(int recno, NOBEL *nbook);

int main() {
    int rno; 
    NOBEL nbook;

    // 1. 임시 파일 빌드
    fp = fopen("booklist.dat", "wb");
    if (fp == NULL) {
        puts("파일 쓰기 에러");
        exit(1);
    }
    datawrite(0, "염소의 축제", "마리오 바르가스 요사", 2010);
    datawrite(1, "기억이 나를 본다", "토마스 트란스트뢰메르", 2011);
    datawrite(2, "사부님은 갈수록 유머러스해진다", "모옌", 2012);
    fclose(fp);

    // 2. 원하는 위치 임의 쿼리
    fp = fopen("booklist.dat", "rb");
    if (fp == NULL) {
        puts("파일 읽기 에러");
        exit(1);
    }
    
    printf("출력할 레코드 번호를 입력 하세요 (0~2): ");
    if (scanf("%d", &rno) != 1) {
        puts("입력 오류");
        fclose(fp);
        return 1;
    }
    
    if (rno < 0 || rno > 2) {
        printf("잘못된 레코드 번호입니다.\n");
    } else {
        dataread(rno, &nbook);
    }
    
    fclose(fp);
    return 0;
}

void datawrite(int recno, char *title, char *author, int year)
{
    fseek(fp, RECLEN * recno, SEEK_SET);
    fprintf(fp, FORMAT, title, author, year);
}

void dataread(int recno, NOBEL *nbook)
{
    // 해당 위치로 다이렉트 fseek 후 파싱 수행
    fseek(fp, recno * RECLEN, SEEK_SET);
    
    fgets(nbook->title, 41, fp);
    fgets(nbook->author, 41, fp);
    if (fscanf(fp, "%d", &nbook->year) != 1) {
        // 파싱
    }
    
    printf("%d\t제목 : %s\n\t작가 : %s\n\t수상년도 : %d \n\n", 
        recno, nbook->title, nbook->author, nbook->year);
}


### 예제 14.9: 파일 크기 동적 획득 및 풀 로딩 (`04_file_io/ex14_9.c`)
파일 포인터를 맨 뒤로 보내어 `ftell`로 파일의 최종 바이트 크기를 구한 뒤, 그만큼 메모리를 `malloc`하여 파일 내용 전체를 한 번에 메모리 버퍼로 읽어 출력하는 동적 할당 연계 응용 기법입니다.

In [ ]:
// 04_file_io/ex14_9.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    FILE *fp;
    char *buf;
    long fsize;

    fp = fopen("ex14_9.c", "rb");
    if (fp == NULL) {
        fprintf(stderr, "파일 열기 실패...\n");
        return 1;
    }
    
    // 파일 끝으로 이동
    fseek(fp, 0, SEEK_END);
    
    // 파일의 현재 위치 값(바이트 오프셋 = 총 파일 바이트수) 획득
    fsize = ftell(fp);
    
    // 버퍼 동적 할당
    buf = (char *)malloc(fsize * sizeof(char) + 1);
    if (buf == NULL) {
        fprintf(stderr, "메모리 할당 실패...\n");
        fclose(fp);
        return 1;
    }
    
    // 포인터를 파일의 맨 처음으로 리셋
    rewind(fp);
    
    // 버퍼 크기만큼 통째로 읽어들이기
    if (fread(buf, 1, fsize, fp) < 1) {
        fprintf(stderr, "파일 읽기 오류...\n");
        free(buf);
        fclose(fp);
        return 1;
    }
    
    buf[fsize] = '\0';
    fclose(fp);
    
    puts("다음은 읽은 파일의 내용입니다.");
    puts(buf);
    
    free(buf);
    return 0;
}